# EDA остатков и календарных признаков

Запусти ноутбук в ядре с готовым датафреймом `df_byn_dates`. Каждая строка соответствует одной дате. Остаток `OST_VAL` указан в млн, календарные признаки берутся из этого же датафрейма.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

flag_columns = [
    "is_weekend", "is_month_start", "is_month_end",
    "is_quarter_start", "is_quarter_end", "is_year_start", "is_year_end",
    "is_holiday", "is_day_before_holiday", "is_day_after_holiday",
]
cyclic_columns = ["weekday_sin", "weekday_cos", "month_sin", "month_cos"]
calendar_columns = [
    "year", "quarter", "month", "day_of_month", "day_of_year",
    "iso_year", "iso_week", "day_of_week", "days_to_month_end",
    *flag_columns, "holiday_name", "days_since_holiday", "days_until_holiday",
    *cyclic_columns,
]
required_columns = ["REP_DT", "OST_VAL", *calendar_columns]

column_names = {name.casefold(): name for name in required_columns}
column_names["date"] = "date"
source = df_byn_dates.rename(columns=lambda name: column_names.get(name.strip().casefold(), name)
                             if isinstance(name, str) else name)
if "REP_DT" not in source.columns:
    if "date" in source.columns:
        source = source.rename(columns={"date": "REP_DT"})
    elif isinstance(source.index, pd.DatetimeIndex):
        source = source.rename_axis("REP_DT").reset_index()

missing_columns = sorted(set(required_columns) - set(source.columns))
if missing_columns:
    raise ValueError("Отсутствуют столбцы: " + ", ".join(missing_columns))

data = source[required_columns].copy()
data["REP_DT"] = pd.to_datetime(data["REP_DT"]).dt.tz_localize(None).dt.normalize()
data["OST_VAL"] = pd.to_numeric(data["OST_VAL"]).astype(float).replace([np.inf, -np.inf], np.nan)
data[flag_columns] = data[flag_columns].astype("boolean")

present_dates = pd.DatetimeIndex(data["REP_DT"].dropna().unique()).sort_values()
daily = data.dropna(subset=["REP_DT", "OST_VAL"]).set_index("REP_DT").sort_index()

## 1. Качество данных и основные показатели

Пропуски в календаре считаются между первой и последней датой. Если отчётность ведётся по рабочим дням, обычные выходные тоже попадут в этот список.

In [ ]:
missing_dates = (
    pd.date_range(present_dates[0], present_dates[-1], freq="D").difference(present_dates)
    if len(present_dates)
    else pd.DatetimeIndex([])
)

quality = pd.Series({
    "Строки": len(data),
    "Строки без даты": data["REP_DT"].isna().sum(),
    "Строки без значения OST_VAL": data["OST_VAL"].isna().sum(),
    "Даты со значением OST_VAL": len(daily),
    "Даты без значения OST_VAL": len(present_dates) - len(daily),
    "Календарные дни без строк": len(missing_dates),
}, name="Количество")
display(quality.to_frame())

if len(missing_dates):
    display(pd.DataFrame({"Первые 15 дат без строк": missing_dates[:15].strftime("%Y-%m-%d")}))
if daily.empty:
    raise ValueError("Для графиков нужна хотя бы одна строка с датой и числовым OST_VAL.")

summary = daily["OST_VAL"].describe(percentiles=[.05, .25, .5, .75, .95])
display(summary.round(3).to_frame("Дневной остаток, млн"))

## 2. Динамика и распределение остатков

Изменения за день рассчитаны только для пар соседних календарных дат с остатком. Скользящая медиана использует доступные наблюдения в окне за 30 календарных дней.

In [ ]:
balance = daily["OST_VAL"]
is_next_day = daily.index.to_series().diff().eq(pd.Timedelta(days=1))
daily_change = balance.diff().loc[is_next_day]

fig, axes = plt.subplot_mosaic(
    [["trend", "trend"], ["distribution", "change"]],
    figsize=(13, 7), layout="constrained",
)

ax = axes["trend"]
ax.scatter(balance.index, balance, s=10, alpha=.45, label="Дневной остаток")
ax.plot(balance.rolling("30D").median(), color="tab:orange", label="Медиана за 30 дней")
ax.set(title="Динамика остатков", ylabel="Остаток, млн")
ax.legend()

ax = axes["distribution"]
ax.hist(balance, bins=30, edgecolor="white")
ax.axvline(balance.median(), color="tab:orange", label="Медиана")
ax.set(title="Распределение дневных остатков", xlabel="Остаток, млн", ylabel="Даты")
ax.legend()

ax = axes["change"]
if daily_change.empty:
    ax.text(.5, .5, "Нет пар соседних дат", transform=ax.transAxes, ha="center", va="center")
else:
    ax.scatter(daily_change.index, daily_change, s=13, alpha=.6)
    ax.axhline(0, color="0.4", linewidth=1)
ax.set(title="Изменение остатка за день", ylabel="Изменение, млн")
ax.tick_params(axis="x", labelrotation=25)

for ax in axes.values():
    ax.grid(axis="y", alpha=.2)
plt.show()

## 3. Календарная сезонность

Точки показывают медиану по доступным датам, полоса — интервал между 25-м и 75-м процентилями. В таблицах `Даты` — размер группы.

In [ ]:
def summarize_groups(frame, by):
    return frame.groupby(by, observed=True)["OST_VAL"].agg(
        dates="size",
        median="median",
        q25=lambda values: values.quantile(.25),
        q75=lambda values: values.quantile(.75),
    )


def plot_profile(ax, stats, title):
    x = stats.index.to_numpy(dtype=float)
    ax.plot(x, stats["median"].to_numpy(dtype=float), "o-", markersize=3)
    ax.fill_between(
        x, stats["q25"].to_numpy(dtype=float), stats["q75"].to_numpy(dtype=float),
        alpha=.17,
    )
    ax.set(title=title, ylabel="Медиана остатков, млн")
    ax.grid(axis="y", alpha=.2)


weekdays = ["Пн", "Вт", "Ср", "Чт", "Пт", "Сб", "Вс"]
profiles = [
    ("day_of_week", range(7), "День недели"),
    ("month", range(1, 13), "Месяц"),
    ("day_of_month", range(1, 32), "День месяца"),
    ("days_to_month_end", range(31), "Дней до конца месяца"),
    ("iso_week", range(1, 54), "Номер недели ISO"),
    ("day_of_year", range(1, 367), "День года"),
]
fig, axes = plt.subplots(2, 3, figsize=(14, 7), layout="constrained")
for ax, (feature, domain, title) in zip(axes.flat, profiles):
    stats = summarize_groups(daily, feature).reindex(domain)
    plot_profile(ax, stats, title)
    if feature == "day_of_week":
        ax.set_xticks(range(7), weekdays)
    if feature == "month":
        ax.set_xticks(range(1, 13))
plt.show()

quarter_stats = summarize_groups(daily, ["year", "quarter"])
display(quarter_stats[["dates", "median"]].unstack("quarter").rename(
    columns={"dates": "Даты", "median": "Медиана, млн"}, level=0,
).rename_axis(index="Год").round(3))

weekly_stats = summarize_groups(daily, ["iso_year", "iso_week"])
display(weekly_stats[["dates", "median"]].tail(8).rename(
    columns={"dates": "Даты", "median": "Медиана, млн"},
).rename_axis(index=["Год ISO", "Неделя ISO"]).round(3))

## 4. Выходные, праздники и границы периода

Каждый флаг сравнивает две группы дат. Разница рассчитана как медиана при `True` минус медиана при `False`. Количество дат помогает оценить, насколько велика каждая группа.

In [ ]:
flag_labels = {
    "is_weekend": "Суббота или воскресенье",
    "is_month_start": "Начало месяца",
    "is_month_end": "Конец месяца",
    "is_quarter_start": "Начало квартала",
    "is_quarter_end": "Конец квартала",
    "is_year_start": "Начало года",
    "is_year_end": "Конец года",
    "is_holiday": "Праздник",
    "is_day_before_holiday": "День перед праздником",
    "is_day_after_holiday": "День после праздника",
}

comparisons = []
for flag, label in flag_labels.items():
    stats = summarize_groups(daily, flag).reindex([False, True])
    comparisons.append({
        "Признак": label,
        "Даты True": int(stats["dates"].fillna(0).loc[True]),
        "Даты False": int(stats["dates"].fillna(0).loc[False]),
        "Медиана True, млн": stats["median"].loc[True],
        "Медиана False, млн": stats["median"].loc[False],
        "Разница, млн": stats["median"].loc[True] - stats["median"].loc[False],
    })

flag_stats = pd.DataFrame(comparisons).set_index("Признак")
display(flag_stats.round(3))

fig, ax = plt.subplots(figsize=(9, 4.5), layout="constrained")
difference = flag_stats["Разница, млн"].dropna().sort_values()
if difference.empty:
    ax.text(.5, .5, "Нет флагов с двумя группами дат", transform=ax.transAxes,
            ha="center", va="center")
else:
    ax.barh(difference.index, difference, color="tab:blue")
    ax.axvline(0, color="0.4", linewidth=1)
ax.set(title="Разница медиан по календарным флагам", xlabel="Разница, млн")
ax.grid(axis="x", alpha=.2)
plt.show()

## 5. Остатки рядом с праздниками

Каждая дата попадает в группу по ближайшему празднику. Ноль — сам праздник, минус — дни до него, плюс — дни после. При равном расстоянии используется предыдущий праздник.

In [ ]:
since = daily["days_since_holiday"]
until = daily["days_until_holiday"]
use_previous = until.isna() | (since.notna() & since.le(until))
offset = since.where(use_previous, -until)
around = daily.assign(holiday_offset=offset).loc[offset.between(-7, 7)]
holiday_profile = summarize_groups(around, "holiday_offset").reindex(range(-7, 8))

fig, (ax_median, ax_count) = plt.subplots(
    2, 1, figsize=(10, 6), sharex=True, layout="constrained",
)
plot_profile(ax_median, holiday_profile, "Остаток в пределах недели от праздника")
ax_count.bar(holiday_profile.index, holiday_profile["dates"].fillna(0))
ax_count.set(xlabel="Дней от ближайшего праздника", ylabel="Даты")
ax_count.set_xticks(range(-7, 8))
for ax in (ax_median, ax_count):
    ax.axvline(0, color="tab:orange", linestyle="--", linewidth=1)
plt.show()

holiday_by_name = summarize_groups(daily, "holiday_name").sort_values("dates", ascending=False)
display(holiday_by_name[["dates", "median"]].rename(
    columns={"dates": "Даты", "median": "Медиана, млн"},
).rename_axis("Праздник").round(3))

## 6. Циклические признаки

Корреляции Пирсона показывают линейную связь каждой компоненты синуса и косинуса с дневным остатком.

In [ ]:
cyclic = daily[cyclic_columns]
cyclic_stats = pd.DataFrame({
    "Даты": cyclic.count(),
    "Корреляция Пирсона": cyclic.assign(OST_VAL=balance).corr()["OST_VAL"].reindex(cyclic_columns),
})
display(cyclic_stats.round(3))